https://python.langchain.com/docs/integrations/chat/

In [1]:
%pip install -qU langchain-openai


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable is not set in the enviroment variables.")
else:
    print("OPENAI_API_KEY is set in the environment variables.")

OPENAI_API_KEY is set in the environment variables.


In [3]:
from langchain_openai import ChatOpenAI

# Select model
model = ChatOpenAI(
    model="gpt-4o-mini" # currently the cheapeast one.
)
# Talk to the model
model.invoke("Hello").content

'Hello! How can I assist you today?'

In [6]:
# provideing more settings to the model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5, # Control randomness of output
    max_tokens=1000, # Control lenght of the output
    top_p=0.96, # trimming the list of possible next words to the most likely ones, based on their probabilities
    frequency_penalty=0, # Control how much to penalize new tokens based on their existing frequency in the text so far
    presence_penalty=0, # Control how much to penalize new tokens based on whether they appear in the text so far
)

# llm.invoke("Capital of USA")

### Prompt Templates

https://python.langchain.com/docs/concepts/prompt_templates/

In [5]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="Translate the following English text to {language}: {input}"
)

chain = prompt | model
# The pipe operator | is used to chain the prompt and model together

chain.invoke({
    "language": "French",
    "input": "I love programming."
}).content

"J'aime la programmation."

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# We can provide a list of messages, and the role that each messages provides.
messages = [
    SystemMessage(content="You are a helpful assistant that answers questions about US History."),
    HumanMessage(content="Who is the second president of the United States?"),
    AIMessage(content="The second president of the United States is George Washington."),
    # Whit AIMessage, the model will asume it already provided that output.
    HumanMessage(content="That answers seems not to be right"),
]

model.invoke(messages).content

'I apologize for the mistake. The second president of the United States is John Adams. He served from 1797 to 1801. George Washington was the first president. Thank you for your understanding!'

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import HumanMessagePromptTemplate, SystemMessagePromptTemplate

# Instead of having just a list by itself we use a ChatPromptTemplate passing it a list to it, with prompt templates, that have assigned roles.

prompt = ChatPromptTemplate([
    SystemMessagePromptTemplate.from_template("You are a {occupation} named {name}. Get into the character and pretend to be this role. Answer questions accordingly."),
    HumanMessagePromptTemplate.from_template("{input}")
])

chain = prompt | model 

chain.invoke({
    "occupation": "old wizard",
    "name": "Gandalf the Grey",
    "input": "Why 42 is the Ultimate Question of Life, the Universe, and Everything?"
}).content

'Ah, young seeker of truth, the number 42 has taken on a rather curious and enigmatic reputation in the realms of thought and fiction. It hails from the mind of the brilliant Douglas Adams, a writer of wit and whimsy, who declared this number as the answer to the Ultimate Question of Life, the Universe, and Everything in his most entertaining tome, "The Hitchhiker\'s Guide to the Galaxy."\n\nHowever, one must ponder: what is the ultimate question to which 42 is the answer? It is said that the question itself remains elusive, a mystery perhaps too profound for mere mortals. Some suggest that the nature of existence, meaning, and purpose is beyond the comprehension of any simple numeric solution. \n\nThus, dear one, while 42 is presented playfully as the answer, I urge you to delve deeper into the mysteries of your own existence, for the true essence of life may lie not in a single number, but in the journey of discovery itself. Embrace the quest for understanding, and you may find that 

In [9]:
# A simplistic example of RAG to come. RAG involves getting relevant information and then augmenting it into the prompt.
prompt = ChatPromptTemplate([
    SystemMessagePromptTemplate.from_template("You are a helpful assistant that answers questions. ONLY use the context provided to answer questions. If the context does not provide the answer, say 'I don't know.' Context: {context}"),
    HumanMessagePromptTemplate.from_template("{input}"),
])

chain = prompt | model

chain.invoke({
    "context": "The capital of Germany is Berlin",
    "input": "What is the capital of USA?"
}).content

"I don't know."